In [2]:
import pandas as pd
import altair as alt
import pubchempy as pcp
import numpy as np
import matplotlib.pyplot as plt

1. Crear dataframe con los datos de las perturbaciones en LINCS.

In [3]:
df = pd.read_csv('GSE70138_Broad_LINCS_pert_info.txt', sep = '\t', usecols = [0, 3]) # Crear df importando datos del archivo (sólo cols de interés), separándolos por una sangría y sin headers por columna
display(df) # Print con mejor visualización

,pert_id,pert_iname
0,BRD-K70792160,10-DEBC
1,BRD-K68552125,phorbol-myristate-acetate
2,BRD-K92301463,"16,16-dimethylprostaglandin-e2"
3,BRD-A29731977,17-hydroxyprogesterone-caproate
4,BRD-K07954936,2-iminobiotin
...,...,...
2165,BRDN0000585417,LACZ
2166,BRDN0000562990,EGFP
2167,BRDN0000585533,LUCIFERASE
2168,BRDN0000563287,EGFP


2. Preprocesar la base de datos eliminando los duplicados, datos vacíos, etc.

In [4]:
df = df.dropna() # Eliminar filas sin información
df = df.drop_duplicates(['pert_id']) # Elimina información duplicada en las columnas indicadas
display(df) # Print con mejor visualización

,pert_id,pert_iname
0,BRD-K70792160,10-DEBC
1,BRD-K68552125,phorbol-myristate-acetate
2,BRD-K92301463,"16,16-dimethylprostaglandin-e2"
3,BRD-A29731977,17-hydroxyprogesterone-caproate
4,BRD-K07954936,2-iminobiotin
...,...,...
2165,BRDN0000585417,LACZ
2166,BRDN0000562990,EGFP
2167,BRDN0000585533,LUCIFERASE
2168,BRDN0000563287,EGFP


3. Contar el número de perturbaciones por molécula.

In [5]:
final = df.value_counts("pert_iname", sort = True) # Cuenta el número de filas asociadas a cada valor único de la columna indicada y los ordena en orden descendente
print(final) # Imprime el resultado

pert_iname
PIK3CA              14
EGFP                12
AKT2                 7
MYC                  7
PLXNA1               7
                    ..
benzylpenicillin     1
benzydamine          1
benzthiazide         1
benzotript           1
zosuquidar           1
Length: 1826, dtype: int64


In [6]:
df_final = final.rename_axis('drug_name').reset_index(name='num_perts') # Convierte la estructura de datos 'final' en un dataframe
display(df_final) # Print con mejor visualización

,drug_name,num_perts
0,PIK3CA,14
1,EGFP,12
2,AKT2,7
3,MYC,7
4,PLXNA1,7
...,...,...
1821,benzylpenicillin,1
1822,benzydamine,1
1823,benzthiazide,1
1824,benzotript,1


4. Identificar el ID de PubChem en los antidepresivos enlistados en SIDER 4.1.

In [7]:
ad = pd.read_csv('drug_atc.tsv', sep = '\t', usecols = [1], header = None) # Crear df importando datos del archivo (sólo cols de interés), separándolos por una sangría y sin headers por columna
display(ad) # Print con mejor visualización

,1
0,A16AA01
1,L03AA03
2,N03AG03
3,L01XD04
4,V03AF03
...,...
1555,J01AA12
1556,J01AA04
1557,S01LA03
1558,C10AC01


In [8]:
ad.rename(columns = {1:'ATC_code'}, inplace = True) # Cambiar nombre a las columnas
display(ad) # Print con mejor visualización

,ATC_code
0,A16AA01
1,L03AA03
2,N03AG03
3,L01XD04
4,V03AF03
...,...
1555,J01AA12
1556,J01AA04
1557,S01LA03
1558,C10AC01


In [9]:
def antidepressants(i):
    if str(i[0:4]) == 'N06A': # ATC_code para antidepresivos es 'N06A...'
        return True
    else:
        return False

In [10]:
ad['AD'] = ad.ATC_code.map(antidepressants) # Utiliza la función con los datos de la columna 'ATC_code' y asigna True/False según si son antidepresivos o no
display(ad) # Print con mejor visualización

,ATC_code,AD
0,A16AA01,False
1,L03AA03,False
2,N03AG03,False
3,L01XD04,False
4,V03AF03,False
...,...,...
1555,J01AA12,False
1556,J01AA04,False
1557,S01LA03,False
1558,C10AC01,False


In [11]:
ad = ad[ad['AD'] == True] # Elimina todos los datos que sean False en la columna 'AD', es decir, los que no son antidepresivos
ad = ad.drop(['AD'], axis = 1) # Elimina la columna que define si es un antidepresivo o no
display(ad) # Print con mejor visualización

,ATC_code
51,N06AX12
159,N06AA09
165,N06AA17
307,N06AB04
308,N06AB10
319,N06AA04
355,N06AA01
419,N06AA16
422,N06AA12
493,N06AB03


5. Identificar el nombre del antidepresivo enlistado en SIDER 4.1 según su código ATC.

In [12]:
nombres = pd.read_csv('names_ATC.csv', sep = ',', usecols = ['atc_code', 'atc_name']) # Crear df importando datos del archivo (sólo cols de interés), separándolos por una sangría y sin headers por columna
display(nombres) # Print con mejor visualización

,atc_code,atc_name
0,A,ALIMENTARY TRACT AND METABOLISM
1,A01,STOMATOLOGICAL PREPARATIONS
2,A01A,STOMATOLOGICAL PREPARATIONS
3,A01AA,Caries prophylactic agents
4,A01AA01,sodium fluoride
...,...,...
6966,V10XX01,sodium phosphate (32P)
6967,V10XX02,ibritumomab tiuxetan (90Y)
6968,V10XX03,radium (223Ra) dichloride
6969,V10XX04,lutetium (177Lu) oxodotreotide


In [13]:
names = nombres.loc[nombres['atc_code'].isin(ad['ATC_code'])] # Localizar todas las columnas en el dataframe directorio con los códigos ATC para encontrar los nombres de dichos medicamentos
names = names.dropna() # Eliminar filas sin información
names = names.drop_duplicates(['atc_code', 'atc_name']) # Elimina información duplicada en las columnas indicadas
display(names) # Print con mejor visualización

,atc_code,atc_name
5284,N06AA01,desipramine
5285,N06AA02,imipramine
5288,N06AA04,clomipramine
5291,N06AA06,trimipramine
5295,N06AA09,amitriptyline
5297,N06AA10,nortriptyline
5299,N06AA11,protriptyline
5300,N06AA12,doxepin
5306,N06AA16,dosulepin
5307,N06AA17,amoxapine


In [14]:
def verifyCID(drug_name): # Asignar o corroborar 'pubchem cid'
    for c in pcp.get_compounds(str(drug_name), 'name'): # Buscar el CID de PubChem según el nombre del compuesto
        return c.cid # Devolver el CID de PubChem

In [15]:
names['pubchem_cid'] = names.atc_name.map(verifyCID) # Utiliza la función 'verifyCID' para buscar los CIDs con los datos de los nombres de los compuestos en el df, y genera una nueva columna
display(names) # Print con mejor visualización

,atc_code,atc_name,pubchem_cid
5284,N06AA01,desipramine,2995
5285,N06AA02,imipramine,3696
5288,N06AA04,clomipramine,2801
5291,N06AA06,trimipramine,5584
5295,N06AA09,amitriptyline,2160
5297,N06AA10,nortriptyline,4543
5299,N06AA11,protriptyline,4976
5300,N06AA12,doxepin,667477
5306,N06AA16,dosulepin,5284550
5307,N06AA17,amoxapine,2170


In [16]:
display(df_final) # Print con mejor visualización

,drug_name,num_perts
0,PIK3CA,14
1,EGFP,12
2,AKT2,7
3,MYC,7
4,PLXNA1,7
...,...,...
1821,benzylpenicillin,1
1822,benzydamine,1
1823,benzthiazide,1
1824,benzotript,1


In [17]:
final_graph = df_final.loc[df_final['drug_name'].isin(names['atc_name'])] # Encontrar las columnas con los valores 
display(final_graph)

,drug_name,num_perts
153,maprotiline,1
176,nortriptyline,1
224,mirtazapine,1
230,milnacipran,1
240,mianserin,1
251,nefazodone,1
278,moclobemide,1
305,fluvoxamine,1
340,fluoxetine,1
481,imipramine,1


5. Graficar el resultado

In [28]:
alt.Chart(final_graph,
        title = alt.TitleParams('Número de perturbaciones por antidepresivo en LINCS NIH (fase 2)', anchor='middle')).mark_bar().encode(
        x = alt.X('drug_name', title = "Nombre del antidepresivo (STITCH database)"),
        y = alt.Y('num_perts', title = "Número de perturbaciones por antidepresivo"),
        color = alt.value('blue'))

alt.Chart(...)